## EDA Notebook for DTSC 4302

In [ ]:
# non built-in libraries
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# built-in libraries
import json
import os
from pathlib import Path
import sys
import datetime
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# personally-defined modules
sys.path.append(os.path.join(str(Path.cwd()), "../"))  
from scripts.data_download import download_files_from_fld

# install reqd. datasets from Google Drive
folder_id = "1P2FRAkPrqL2nn2MNMyd4ilWbXNS_kkKD" 
data_path = os.path.join(str(Path.cwd()), "../data")
download_files_from_fld(folder_id, data_path)

# constants
START_DATE = datetime.datetime(2013, 10, 1)  # init start date of analysis to first day of FY 2014
END_DATE = datetime.datetime(2024, 9, 30)  # init end date of analysis to last day of FY 2024

# reading in data
sent_df = pd.read_csv(os.path.join(data_path, "sentencing_data_cleaned.csv"), low_memory=False)
districts_gdf = gpd.read_file(os.path.join(data_path, "us_district_cts_bounds.geojson"))
districts_gdf = districts_gdf.rename(columns={c: c.upper() for c in districts_gdf.columns})
districts_gdf = districts_gdf[["NAME", "DISTRICT_N", "GEOMETRY"]]

Violin plots showing distribution of sentence lengths over time, by race (white vs black) and gender.

In [ ]:
df = sent_df[["RACE", "SEX", "FISCAL_YR", "MNTHS_PRSN_NO_ALT"]].copy()

# keep only prison sentences so log is defined and interpretation is conditional on incarceration
df = df[df["MNTHS_PRSN_NO_ALT"] > 0].copy()

# log transform sentence lengths
df["LOG_SENTENCE"] = np.log(df["MNTHS_PRSN_NO_ALT"])

# convert fiscal year to string so seaborn treats it as categorical
df["FISCAL_YR"] = df["FISCAL_YR"].astype(str)

# separate dataframes for each subplot
df_race = df[df["RACE"].isin(["White", "Black"])].copy()
df_sex = df[df["SEX"].isin(["Male", "Female"])].copy()

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# white vs black subplot
sns.violinplot(
    data=df_race,
    x="FISCAL_YR",
    y="LOG_SENTENCE",
    hue="RACE",
    split=True,
    inner="quartile",
    cut=0,
    bw_adjust=0.8,
    ax=axes[0]
)

axes[0].set_title("Distribution of Log Sentence Length by Race and Fiscal Year", fontsize=16, pad=12)
axes[0].set_xlabel("")
axes[0].set_ylabel("Log Sentence Length (Months)")
axes[0].grid(False)
axes[0].legend(title="")

# -male vs female subplot
sns.violinplot(
    data=df_sex,
    x="FISCAL_YR",
    y="LOG_SENTENCE",
    hue="SEX",
    split=True,
    inner="quartile",
    cut=0,
    bw_adjust=0.8,
    ax=axes[1]
)

axes[1].set_title("Distribution of Log Sentence Length by Gender and Fiscal Year", fontsize=16, pad=12)
axes[1].set_xlabel("")
axes[1].set_ylabel("Log Sentence Length (Months)")
axes[1].grid(False)
axes[1].legend(title="")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.text(
    0.01, -0.03,
    "Note: The sentence length shown in the graph is conditional on the offender receiving a prison sentence, so offenders with no prison sentence are excluded. The long tails at the bottom represent sentences of a few days. " \
    "\nImmigration cases, cases with missing sentence lengths, and cases with missing demographic information are excluded.",
    ha="left",
    fontsize=10
)

plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

In [ ]:
df = sent_df[["RACE", "SEX", "MNTHS_VARIANCE/DEPARTURE"]].copy()

# create categorical variable for whether sentence is above, below, or within guidelines
df["guideline_position"] = pd.cut(
    df["MNTHS_VARIANCE/DEPARTURE"],
    bins=[-np.inf, -1e-12, 1e-12, np.inf],
    labels=["Below", "Within", "Above"]
)

df = df.dropna(subset=["guideline_position"])

# count observations in each race-sex-guideline cell
pct_df = (
    df.groupby(["RACE", "SEX", "guideline_position"], observed=False)
      .size()
      .reset_index(name="count")
)

# convert counts to percentages within each race-sex group
pct_df["percent"] = (
    pct_df["count"] /
    pct_df.groupby(["RACE", "SEX"], observed=False)["count"].transform("sum")
) * 100

race_order = ["White", "Black", "Hispanic", "Other"]
sex_order = ["Male", "Female"]
positions = ["Below", "Within", "Above"]

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

bar_width = 0.35
x = np.arange(len(race_order))

colors = {
    "Male": "#4C72B0",
    "Female": "#DD8452"
}

for i, pos in enumerate(positions):
    ax = axes[i]
    sub = pct_df[pct_df["guideline_position"] == pos]

    for j, sex in enumerate(sex_order):
        vals = (
            sub[sub["SEX"] == sex]
            .set_index("RACE")
            .reindex(race_order)["percent"]
        )

        ax.bar(
            x + (j - 0.5) * bar_width,
            vals,
            width=bar_width,
            label=sex if i == 2 else None,
            color=colors[sex],
            edgecolor="black"
        )

    ax.set_title(f"{pos} Guidelines")
    ax.set_xticks(x)
    ax.set_xticklabels(race_order)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)


axes[0].set_ylabel("Percent of Group")
axes[2].legend(title="Sex", frameon=False)

# annotate bars with percentages
for ax in axes:
    for bar in ax.patches:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.5,
            f"{height:.1f}",
            ha="center",
            fontsize=8
        )

# set ylim to be 0.8 across all graphs
axes[0].set_ylim(0, 80)

fig.suptitle(
    "Sentencing Relative to Guidelines by Race and Sex",
    fontsize=15,
    y=1.02
)

fig.text(
    0.01, -0.09,
    "Notes: The values are normalized such that the percentages for a given race-sex group sum to 100% across the 3 graphs." \
    "Additionally, this graph does not include immigration offenses or cases where the sex/race of the offender is unknown.",
    ha="left",
    fontsize=10
)

plt.tight_layout()
plt.show()

Pivot table showing the distribution of offense types for each race

In [ ]:
pd.crosstab(sent_df["OFF_TYPE"], sent_df["RACE"], normalize="columns") * 100

Pivot table showing the proportion of offenders (by race and sex) that receive a 5K1.1 downward departure for helping the prosecution convict someone else

In [ ]:
pd.pivot_table(
    sent_df,
    values="5K1.1",
    index="SEX",
    columns="RACE",
    aggfunc="mean"
) * 100

Average variance/departure by race and gender pivot tables

In [ ]:
# average variance/departure by race and gender unconditional on if there was a departure or not (i.e. including cases with 0 variance/departure)
print(
    pd.pivot_table(
        sent_df,
        values="MNTHS_VARIANCE/DEPARTURE",
        index="SEX",
        columns="RACE",
        aggfunc="mean"
    ) 
)

print("\n\n")

# average variance/departure by race and gender conditional on if there was a departure or not (i.e. excluding cases with 0 variance/departure)
print(
    pd.pivot_table(
        sent_df[sent_df["MNTHS_VARIANCE/DEPARTURE"] != 0],
        values="MNTHS_VARIANCE/DEPARTURE",
        index="SEX",
        columns="RACE",
        aggfunc="mean"
    ) 
)

Pivot table showing the percentage of each race that pleas guilty

In [ ]:
(1 - pd.pivot_table(
    sent_df,
    values="TRIAL_FLAG",
    index="SEX",
    columns="RACE",
    aggfunc="mean"
)) * 100

Pivot table showing most common offense types committed for different offender education levels

In [ ]:
(pd.crosstab(sent_df["EDUCATION"], sent_df["OFF_TYPE"], normalize="index") * 100).round(2)

Graph showing how much the midpoint of the guideline range explains sentencing lengths across fiscal years 2014-2024, separated by race. 

Maybe we can also consider adding a subplot for gender.

In [ ]:
df = sent_df[["FISCAL_YR", "RACE", "MNTHS_PRSN_NO_ALT", "GL_MIN", "GL_MAX"]].copy().dropna()

df = df[(df["MNTHS_PRSN_NO_ALT"] >= 0) & (df["GL_MIN"] >= 0) & (df["GL_MAX"] >= 0)]
df = df[df["GL_MAX"] < 1000]

# guideline midpoint
df["GLMID"] = (df["GL_MIN"] + df["GL_MAX"]) / 2

# logs
df["log_sent"] = np.log1p(df["MNTHS_PRSN_NO_ALT"])
df["log_glmid"] = np.log1p(df["GLMID"])

def compute_r2(subdf):
    X = sm.add_constant(subdf["log_glmid"])
    y = subdf["log_sent"]
    return sm.OLS(y, X).fit().rsquared

def bootstrap_r2_ci(subdf, n_boot=300, random_state=42):
    rng = np.random.default_rng(random_state)
    n = len(subdf)

    r2_hat = compute_r2(subdf)
    boot = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = compute_r2(subdf.iloc[idx])

    return pd.Series({
        "R2": r2_hat,
        "R2_lower": np.percentile(boot, 2.5),
        "R2_upper": np.percentile(boot, 97.5),
        "N": n
    })


results = []

for (year, race), subdf in df.groupby(["FISCAL_YR", "RACE"]):
    if len(subdf) >= 30:
        out = bootstrap_r2_ci(subdf)
        out["FISCAL_YR"] = year
        out["RACE"] = race
        results.append(out)

r2_df = pd.DataFrame(results)


# plotting code
plt.figure(figsize=(8, 5))

race_order = ["White", "Black", "Hispanic", "Other"]
colors = {
    "White": "#4C72B0",
    "Black": "#DD8452",
    "Hispanic": "#55A868",
    "Other": "#C44E52"
}

for race in race_order:
    sub = r2_df[r2_df["RACE"] == race].sort_values("FISCAL_YR")

    if len(sub) == 0:
        continue

    color = colors.get(race, None)

    # line
    plt.plot(
        sub["FISCAL_YR"],
        sub["R2"],
        linewidth=2,
        color=color,
        label=race
    )

    # shaded CI
    plt.fill_between(
        sub["FISCAL_YR"],
        sub["R2_lower"],
        sub["R2_upper"],
        color=color,
        alpha=0.15
    )

plt.ylim(0, 1)
plt.xlabel("Fiscal Year")
plt.ylabel("R²")
plt.title("What % of Sentence Variation is Explained by the Guideline Range?")
plt.figtext(
    0.05, -0.09,
    "Note: R² is computed from a regression of log sentence length on log guideline midpoint. Shaded areas represent " \
    "\n95% confidence intervals computed via bootstrapping. Immigration cases, cases with missing sentence lengths, \nand cases with missing demographic information are excluded.",
    ha="left",
    fontsize=10
)
plt.legend(frameon=False, fontsize=10)
plt.grid(False)
plt.show()

Average sentence length by country of citizenship choropleth map

In [ ]:
df = sent_df[["CTRY_OF_CTZNSHP", "MNTHS_PRSN_NO_ALT"]].dropna()

country_avg = (
    df.groupby("CTRY_OF_CTZNSHP")["MNTHS_PRSN_NO_ALT"]
    .mean()
    .reset_index()
)

country_avg.columns = ["country", "avg_sentence"]

fig = px.choropleth(
    country_avg,
    locations="country",
    color="avg_sentence",
    color_continuous_scale="viridis",
    projection="natural earth",
    title="Average Sentence Length by Country of Citizenship",
)

fig.update_layout(
    coloraxis_colorbar_title="Avg Sentence (Months)",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

Animated choropleth maps showing the distribution of average sentence length and incarceration rate by district and fiscal year.

In [ ]:
# build district-year summary dataframe
summary_df = (
    sent_df[["DIST_CRT", "FISCAL_YR", "RECIEVED_PRSN_FLAG", "MNTHS_PRSN_NO_ALT"]]
    .dropna(subset=["DIST_CRT", "FISCAL_YR"])
    .groupby(["DIST_CRT", "FISCAL_YR"], as_index=False)
    .agg(
        incarceration_rate=("RECIEVED_PRSN_FLAG", "mean"),
        avg_sentence=("MNTHS_PRSN_NO_ALT", lambda x: x[x > 0].mean())
    )
)

summary_df["incarceration_rate"] *= 100

# merge with district dataframe 
map_df = summary_df.merge(
    districts_gdf[["NAME", "GEOMETRY"]],
    left_on="DIST_CRT",
    right_on="NAME",
    how="left"
)

map_gdf = gpd.GeoDataFrame(map_df, geometry="GEOMETRY")
district_geojson = json.loads(districts_gdf.set_geometry("GEOMETRY").to_json())

# --------------------------------------------------
# 3. Setup years and color ranges
# --------------------------------------------------
years = sorted(map_gdf["FISCAL_YR"].dropna().unique())

incar_vmin = map_gdf["incarceration_rate"].min()
incar_vmax = map_gdf["incarceration_rate"].max()

sent_vmin = map_gdf["avg_sentence"].min()
sent_vmax = map_gdf["avg_sentence"].max()

# create subplot figure
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "choropleth"}, {"type": "choropleth"}]],
    subplot_titles=("Incarceration Rate (%)", "Average Sentence Length")
)

# init year traces
initial_year = years[0]
df0 = map_gdf[map_gdf["FISCAL_YR"] == initial_year]

fig.add_trace(
    go.Choropleth(
        geojson=district_geojson,
        locations=df0["DIST_CRT"],
        z=df0["incarceration_rate"],
        featureidkey="properties.NAME",
        colorscale="Blues",
        zmin=incar_vmin,
        zmax=incar_vmax,
        colorbar=dict(title="Prison %", x=0.45),
        marker_line_width=0.3
    ),
    row=1, col=1
)

fig.add_trace(
    go.Choropleth(
        geojson=district_geojson,
        locations=df0["DIST_CRT"],
        z=df0["avg_sentence"],
        featureidkey="properties.NAME",
        colorscale="Viridis",
        zmin=sent_vmin,
        zmax=sent_vmax,
        colorbar=dict(title="Months", x=1.02),
        marker_line_width=0.3
    ),
    row=1, col=2
)

# create animation frames by fiscal year
frames = []

for year in years:
    dff = map_gdf[map_gdf["FISCAL_YR"] == year]

    frames.append(
        go.Frame(
            name=str(year),
            data=[
                go.Choropleth(
                    geojson=district_geojson,
                    locations=dff["DIST_CRT"],
                    z=dff["incarceration_rate"],
                    featureidkey="properties.NAME",
                    colorscale="Blues",
                    zmin=incar_vmin,
                    zmax=incar_vmax,
                    marker_line_width=0.3
                ),
                go.Choropleth(
                    geojson=district_geojson,
                    locations=dff["DIST_CRT"],
                    z=dff["avg_sentence"],
                    featureidkey="properties.NAME",
                    colorscale="Viridis",
                    zmin=sent_vmin,
                    zmax=sent_vmax,
                    marker_line_width=0.3
                )
            ]
        )
    )

fig.frames = frames

# layound and animation controls
fig.update_layout(
    title_text="District Sentencing Outcomes Over Time",
    width=1200,
    height=550,
    geo=dict(
        fitbounds="locations",
        visible=False
    ),
    geo2=dict(
        fitbounds="locations",
        visible=False
    ),
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.5,
        "y": 1.12,
        "xanchor": "center",
        "yanchor": "top",
        "buttons": [
            {
                "label": "Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 800, "redraw": True},
                        "transition": {"duration": 300},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }
                ]
            }
        ]
    }],
    sliders=[{
        "active": 0,
        "x": 0.1,
        "y": -0.05,
        "len": 0.8,
        "currentvalue": {"prefix": "Fiscal Year: "},
        "steps": [
            {
                "label": str(year),
                "method": "animate",
                "args": [
                    [str(year)],
                    {
                        "frame": {"duration": 500, "redraw": True},
                        "transition": {"duration": 300},
                        "mode": "immediate"
                    }
                ]
            }
            for year in years
        ]
    }]
)

fig.show()

Average variance/departure by district and year choropleth

In [ ]:
avg_df = (
    sent_df[["DIST_CRT", "FISCAL_YR", "MNTHS_VARIANCE/DEPARTURE"]]
    .dropna()
    .groupby(["DIST_CRT", "FISCAL_YR"], as_index=False)["MNTHS_VARIANCE/DEPARTURE"]
    .mean()
    .rename(columns={"MNTHS_VARIANCE/DEPARTURE": "Average Departure/Variance"})
)

map_df = avg_df.merge(
    districts_gdf[["NAME", "GEOMETRY"]],
    left_on="DIST_CRT",
    right_on="NAME",
    how="left"
)

map_gdf = gpd.GeoDataFrame(map_df, geometry="GEOMETRY")
district_geojson = json.loads(districts_gdf.set_geometry("GEOMETRY").to_json())

fig = px.choropleth(
    map_gdf,
    geojson=district_geojson,
    locations="DIST_CRT",
    featureidkey="properties.NAME",
    color="Average Departure/Variance",
    animation_frame="FISCAL_YR",
    color_continuous_scale="Viridis",
    title="Average Variance/Departure by District and Year"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.show()

Bar graphs showing both incarceration rate and average sentence length (conditional on incarceration) by race and sex

In [ ]:
def annotate_bars(ax, fmt="{:.1f}", offset=0.01):
    """
    Annotate bar containers in a matplotlib axis.
    
    fmt: format string for values
    offset: vertical offset as a fraction of y-axis range
    """
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min

    for container in ax.containers:
        for bar in container:
            height = bar.get_height()
            if np.isnan(height):
                continue

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + offset * y_range,
                fmt.format(height),
                ha="center",
                va="bottom",
                fontsize=12.5,
                fontweight="bold"
            )   


df = sent_df[["RACE", "SEX", "RCVD_PRIS_SENT_ELGB_PROBAT", "MNTHS_PRSN_NO_ALT"]].copy()
df["RACE"] = pd.Categorical(df["RACE"], categories=["White", "Black", "Hispanic", "Other"], ordered=True)
df["SEX"] = pd.Categorical(df["SEX"], categories=["Male", "Female"], ordered=True)

# plot incarceration rates among probation-eligible offenders
incarceration_df = df.dropna(subset=["RCVD_PRIS_SENT_ELGB_PROBAT"]).copy()

incarceration_rates = (
    incarceration_df
    .groupby(["RACE", "SEX"], observed=False)["RCVD_PRIS_SENT_ELGB_PROBAT"]
    .mean()
    .mul(100)
    .unstack()
)

# plot average sentence length among those who received prison sentences
prison_df = df[df["MNTHS_PRSN_NO_ALT"] > 0].copy()

avg_sentence = (
    prison_df
    .groupby(["RACE", "SEX"], observed=False)["MNTHS_PRSN_NO_ALT"]
    .mean()
    .unstack()
)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), constrained_layout=True)

# Left subplot: incarceration percent
incarceration_rates.plot(
    kind="bar",
    ax=axes[0],
    width=0.75,
    edgecolor="black"
)

axes[0].set_title("% Receiving Prison Among Probation-Eligible Offenders", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Percent Receiving Prison", fontweight="bold", fontsize=14)
axes[0].get_legend().remove()  # we only need legend for one graph
axes[0].grid(False)
axes[0].set_axisbelow(True)

# Right subplot: average sentence length conditional on prison
avg_sentence.plot(
    kind="bar",
    ax=axes[1],
    width=0.75,
    edgecolor="black"
)

# annotate bars
annotate_bars(axes[0], fmt="{:.1f}")   # percent plot
annotate_bars(axes[1], fmt="{:.1f}")   # months plot

axes[1].set_title("Average Prison Sentence Length Conditional on Receiving Prison", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Average Sentence Length (Months)", fontweight="bold", fontsize=14)
axes[1].legend(title="Gender", frameon=False)
axes[1].grid(False)
axes[1].set_axisbelow(True)

# remove y ticks
axes[0].set_yticks([])
axes[1].set_yticks([])

# remove top and right spines
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="x", rotation=0)

# make xticks bold
for ax in axes:
    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

fig.suptitle(
    "Sentencing Outcomes by Race and Gender",
    fontsize=15,
    y=1.06,
    fontweight="bold"
)

fig.text(
    0.01, -0.08,
    "Notes: The left graph shows the percentage of probation-eligible offenders who received prison sentences, while the right graph shows the average prison sentence length among those \nwho received prison sentences. The sample is limited to cases with non-missing race, gender, and sentencing information, and does not include immigration offenses.",
    ha="left",
    fontsize=10
)

plt.show()

Heatmap of average variance/departure by CHC and offense level, faceted by race

In [ ]:
df = sent_df[["RACE", "CHC", "OL", "MNTHS_VARIANCE/DEPARTURE"]]

# this filter can be removed if we dont want to look at departures conditional on the offender getting a departure
df = df[df["MNTHS_VARIANCE/DEPARTURE"] != 0]

race_order = ["White", "Black", "Hispanic", "Other"]
df["RACE"] = pd.Categorical(df["RACE"], categories=race_order, ordered=True)

# Bin OLs
df["OL_BIN"] = pd.cut(
    df["OL"],
    bins=[1, 9, 18, 27, 36, 43],
    labels=["1-8", "9-17", "18-26", "27-35", "36-43"],
    right=False,
    include_lowest=True
)

# aggregate means and counts
agg = (
    df.groupby(["RACE", "CHC", "OL_BIN"], observed=False)["MNTHS_VARIANCE/DEPARTURE"]
      .agg(mean="mean", count="size")
      .reset_index()
)

# # optionally mask cells with a small number of observations
# min_count = 50
# agg.loc[agg["count"] < min_count, "mean"] = np.nan

sns.set_theme(style="white", font_scale=1.0)
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
axes = axes.flatten()

# color range for easy comparison across races
vmin = -120
vmax = 20

for i, race in enumerate(race_order):
    ax = axes[i]
    sub = agg[agg["RACE"] == race]

    heatmap_data = sub.pivot(index="CHC", columns="OL_BIN", values="mean")
    count_data = sub.pivot(index="CHC", columns="OL_BIN", values="count")

    sns.heatmap(
        heatmap_data,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        annot=True,  
        annot_kws={"size": 9},
        fmt=".1f",
        linewidths=0.5,
        linecolor="white",
        cbar=False
    )

    ax.set_title(race)
    if i in [2, 3]:
        ax.set_xlabel("Offense Level Bin")
    else:
        ax.set_xlabel("")
    if i in [0, 2]:
        ax.set_ylabel("CHC")
    else:
        ax.set_ylabel("")

    # make CHC ticks (yaxis) integers and rotate 90 degrees for readability
    ax.set_yticks(ax.get_yticks())
    ax.set_yticklabels([int(tick) for tick in ax.get_yticks()])
    ax.tick_params(axis="y", rotation=0)


fig.suptitle(
    "Average Variance/Departure by CHC and Binned OL",
    fontsize=16,
    y=0.95
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Incarceration rate and average sentence length by age group.

In [ ]:
df = sent_df.copy()

df = df[["AGE", "RCVD_PRIS_SENT_ELGB_PROBAT", "MNTHS_PRSN_NO_ALT"]].dropna(subset=["AGE"])

# bin age to reduce noise in plot
age_bins = list(range(16, 93, 5)) 
age_labels = [f"{age_bins[i]}–{age_bins[i+1]-1}" for i in range(len(age_bins)-1)]

df["AGE_BIN"] = pd.cut(
    df["AGE"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

# incarceration rate (among eligible)
incarceration = df.groupby("AGE_BIN")["RCVD_PRIS_SENT_ELGB_PROBAT"].mean() * 100

# Average sentence length conditional on offender receiving prison
avg_sentence = (
    df[df["MNTHS_PRSN_NO_ALT"] > 0]
    .groupby("AGE_BIN")["MNTHS_PRSN_NO_ALT"]
    .mean()
)

# combine dataframes together
combined = pd.concat([incarceration, avg_sentence], axis=1)
combined.columns = ["incarceration_rate", "avg_sentence"]
combined = combined.dropna()

# background age distribution
age_counts = df["AGE_BIN"].value_counts().sort_index()
age_density = age_counts / age_counts.max()

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11
})

fig, ax1 = plt.subplots(figsize=(10, 5))

# subtle background bars to show relative sample size
ax1.bar(
    combined.index,
    age_density.loc[combined.index] * combined["incarceration_rate"].max() * 0.30,
    color="gray",
    alpha=0.10,
    width=0.85,
    zorder=0
)

# left axis: incarceration %
ax1.plot(
    combined.index,
    combined["incarceration_rate"],
    color="#4C72B0",
    linewidth=3,
    label="Incarceration Rate",
    zorder=3
)
ax1.set_ylabel("% Eligible for Probation that Received Prison", color="#4C72B0")
ax1.tick_params(axis='y', labelcolor="#4C72B0")

# right axis: sentence length
ax2 = ax1.twinx()
ax2.plot(
    combined.index,
    combined["avg_sentence"],
    color="#DD8452",
    linewidth=3,
    label="Avg Sentence Length",
    zorder=3
)
ax2.set_ylabel("Average Sentence Length (Months)", color="#DD8452")
ax2.tick_params(axis='y', labelcolor="#DD8452")

# x-axis
ax1.set_xlabel("Age Group")
ax1.set_xticks(range(len(combined.index)))
ax1.set_xticklabels(combined.index, rotation=45)

ax1.grid(axis="y", alpha=0.3)
ax1.set_axisbelow(True)

# remove top spines
for ax in [ax1, ax2]:
    ax.spines["top"].set_visible(False)

# Combined legend
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines_1 + lines_2,
    labels_1 + labels_2,
    frameon=False,
    loc="upper left"
)

# increase y limits slightly for better annotation spacing
ax1.set_ylim(0, combined["incarceration_rate"].max() * 1.25)
ax2.set_ylim(0, combined["avg_sentence"].max() * 1.25)

# footnote
fig.text(
    0.01, -0.09,
    "Notes: The histogram plot on the x-axis shows the relative sample size for each age group and is not related to the y-axis values.\n"
    "Incarceration rate is the percentage of individuals eligible for probation who received a prison sentence.\n"
    "The average sentence length is calculated only among those who received prison sentences. Therefore the sample sizes\n"
    "for the two lines differ, especially in older age groups where fewer offenders receive prison sentences.",
    ha="left",
    fontsize=9
)

plt.title("Sentence Length and Incarceration Rate by Age Group", fontsize=14, pad=15)
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.show()

Distribution of Prison Sentence Length (conditional on incarceration) by Offense Type and Race Group

In [ ]:
df = (
    sent_df[sent_df["MNTHS_PRSN_NO_ALT"] > 0]
    [["OFF_TYPE", "RACE", "MNTHS_PRSN_NO_ALT"]]
    .copy() 
)

df["RACE_GROUP"] = np.where(df["RACE"] == "White", "White", "Non-White")

sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))
ax = sns.boxplot(
    data=df,
    x="OFF_TYPE",
    y="MNTHS_PRSN_NO_ALT",
    hue="RACE_GROUP",
    palette={"White": "#4C72B0", "Non-White": "#DD8452"},
    width=0.7,
    fliersize=2,
    flierprops={"alpha": 0.05}
)


plt.xlabel("")  # it is obvious from the x-ticks that this is offense type
plt.ylabel("Sentence Length (Months)")
plt.title("Distribution of Prison Sentence Length by Offense Type and Race Group", y = 1.09)

plt.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.09),  # move legend above plot
    fontsize=9,
    title_fontsize=10,
    ncol=2
)

ax = plt.gca()
# ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

Trial rate, 5K1.1 departure rate, incarceration rate, and sentence length over time and across districts


In [ ]:
df = sent_df.copy()

df["trial"] = (df["TRIAL_FLAG"] == 1).astype(int)
df["fivek"] = (df["5K1.1"] == 1).astype(int)
df["incarceration"] = (df["RECIEVED_PRSN_FLAG"] == 1).astype(int)

# function to compute district-year summary stats
def compute_summary(data, value_col, conditional=None):
    temp = data.copy()
    
    if conditional is not None:
        temp = temp[conditional(temp)]
    
    district_rates = (
        temp.groupby(["FISCAL_YR", "DIST_CRT"])[value_col]
        .mean()
        .reset_index()
    )
    
    summary = (
        district_rates
        .groupby("FISCAL_YR")[value_col]
        .agg(
            mean="mean",
            q25=lambda x: np.percentile(x, 25),
            q75=lambda x: np.percentile(x, 75)
        )
        .reset_index()
    )
    
    return summary

# compute summary stats for each outcome
trial_summary = compute_summary(df, "trial")
fivek_summary = compute_summary(df, "fivek")
incar_summary = compute_summary(df, "incarceration")

sentence_summary = compute_summary(
    df,
    "MNTHS_PRSN_NO_ALT",
    conditional=lambda d: d["MNTHS_PRSN_NO_ALT"] > 0
)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11
})

fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
axes = axes.flatten()

def plot_panel(ax, summary, title, ylabel, color, percent=True):
    y_mean = summary["mean"] * 100 if percent else summary["mean"]
    y_q25 = summary["q25"] * 100 if percent else summary["q25"]
    y_q75 = summary["q75"] * 100 if percent else summary["q75"]
    
    ax.plot(
        summary["FISCAL_YR"],
        y_mean,
        color=color,
        linewidth=2
    )
    
    ax.fill_between(
        summary["FISCAL_YR"],
        y_q25,
        y_q75,
        color=color,
        alpha=0.2
    )
    
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Panels
plot_panel(
    axes[0],
    trial_summary,
    "Trial Rate",
    "Percent Going to Trial",
    "#DD8452"
)

plot_panel(
    axes[1],
    fivek_summary,
    "5K1.1 Departure Rate",
    "Percent Receiving 5K1.1",
    "#4C72B0"
)

plot_panel(
    axes[2],
    sentence_summary,
    "Avg Sentence Length (Conditional on Prison)",
    "Months",
    "#55A868",
    percent=False
)

plot_panel(
    axes[3],
    incar_summary,
    "Incarceration Rate",
    "Percent Receiving Prison",
    "#C44E52"
)

# X labels only on bottom row
for ax in axes[2:]:
    ax.set_xlabel("Fiscal Year")

fig.suptitle(
    "Sentencing Outcomes Over Time and Across Districts",
    fontsize=15,
    y=0.95
)

fig.text(
    0.01, -0.08,
    "Notes: Each line represents the average value of the outcome across districts in a given fiscal year, while the shaded area represents the " \
    "\ninterquartile range across districts. The sample is limited to cases with non-missing sentencing and district information, and does not include " \
    "\nimmigration offenses.",
    ha="left",
    fontsize=10
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Trends in incarceration rate by fiscal year and different demographic variables

In [ ]:
df = sent_df[[
    "FISCAL_YR", "RACE", "SEX", "AGE", "EDUCATION",
    "RECIEVED_PRSN_FLAG"
]].dropna().copy()

# bin age
age_bins = list(range(16, 96, 15))
age_labels = [f"{age_bins[i]}–{age_bins[i+1]-1}" for i in range(len(age_bins)-1)]

df["AGE_BIN"] = pd.cut(
    df["AGE"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True
)


def make_agg(data, group_col):
    return (
        data.groupby(["FISCAL_YR", group_col], observed=False)["RECIEVED_PRSN_FLAG"]
        .mean()
        .mul(100)
        .reset_index(name="incarceration_rate")
    )

agg_race = make_agg(df, "RACE")
agg_sex = make_agg(df, "SEX")
agg_age = make_agg(df, "AGE_BIN")
agg_edu = make_agg(df, "EDUCATION")


def plot_panel(ax, agg, group_col, title, order=None):
    if order is None:
        groups = sorted(agg[group_col].dropna().unique())
    else:
        groups = [g for g in order if g in set(agg[group_col].dropna())]

    for group in groups:
        sub = agg[agg[group_col] == group].sort_values("FISCAL_YR")

        if len(sub) == 0:
            continue

        ax.plot(
            sub["FISCAL_YR"],
            sub["incarceration_rate"],
            linewidth=2,
            label=str(group),
            marker="."
        )

    ax.set_title(title)
    ax.set_xlabel("Fiscal Year")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize="small")


plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9
})

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.flatten()

race_order = ["White", "Black", "Hispanic", "Other"]
sex_order = ["Male", "Female"]
age_order = age_labels

plot_panel(axes[0], agg_race, "RACE", "Race", race_order)
plot_panel(axes[1], agg_sex, "SEX", "Gender", sex_order)
plot_panel(axes[2], agg_age, "AGE_BIN", "Age Group", age_order)
plot_panel(axes[3], agg_edu, "EDUCATION", "Education")

# y-axis labels
axes[0].set_ylabel("Incarceration Rate (%)")
axes[2].set_ylabel("Incarceration Rate (%)")
axes[1].set_ylabel("")
axes[3].set_ylabel("")

axes[2].set_xlabel("Fiscal Year")
axes[3].set_xlabel("Fiscal Year")
axes[0].set_xlabel("")
axes[1].set_xlabel("")

# x-axis ticks
years = sorted(df["FISCAL_YR"].unique())
for ax in axes:
    ax.set_xticks(years)

fig.suptitle(
    "Trends in Incarceration Rates by Demographic Group (Fiscal years 2014-2024)",
    fontsize=15,
    y=1
)

fig.text(
    0.01, -0.02,
    "Note: This graph does not included cases of immigration offenses or cases where the sex, race, age, or education information of the offender is missing. " \
    "Incarceration rate is defined as the percentage of offenders \nreceiving a prison sentence.",
    ha="left",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

Plea vs. Trial sentence length difference by race, offense level, and CHC

In [ ]:
df = sent_df[[
    "CHC", "OL", "RACE",
    "TRIAL_FLAG",
    "MNTHS_PRSN_NO_ALT"
]].dropna().copy()

# bin OL into groups of 5
df["OL_bin"] = (df["OL"] // 5) * 5

# map trial labels
trial_map = {0: "Plea", 1: "Trial"}
df["TRIAL_LABEL"] = df["TRIAL_FLAG"].map(trial_map)

df = df[df["TRIAL_LABEL"].isin(["Plea", "Trial"])]

agg = (
    df.groupby(["RACE", "CHC", "OL_bin", "TRIAL_LABEL"])["MNTHS_PRSN_NO_ALT"]
    .mean()
    .reset_index()
)

pivot = agg.pivot_table(
    index=["RACE", "CHC", "OL_bin"],
    columns="TRIAL_LABEL",
    values="MNTHS_PRSN_NO_ALT"
).reset_index()

pivot["DIFF"] = pivot["Trial"] - pivot["Plea"]

# plotting cocde
races = sorted(pivot["RACE"].unique())[:4]  # ensure max 4 for 2x2

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True, sharex=True)
axes = axes.flatten()

# global color scale for easy comparison
vmin = pivot["DIFF"].min()
vmax = pivot["DIFF"].max()

for i, (ax, race) in enumerate(zip(axes, races)):
    sub = pivot[pivot["RACE"] == race]

    heat = sub.pivot(
        index="CHC",
        columns="OL_bin",
        values="DIFF"
    )

    # flip CHC so 1 is at bottom
    heat = heat.sort_index(ascending=False)

    # OL labels
    heat.columns = [f"{int(c)}-{int(c+4)}" for c in heat.columns]

    sns.heatmap(
        heat,
        ax=ax,
        cmap="Blues",         
        vmin=vmin,
        vmax=vmax,
        annot=True,          
        fmt=".1f",
        cbar=False            
    )

    ax.set_title(f"Race: {race}")
    if i in [2, 3]:  
        ax.set_xlabel("Offense Level")
    if i in [0, 2]:  
        ax.set_ylabel("CHC")
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=9)



plt.suptitle(
    "Trial vs Plea Sentence Difference by Race\n(Positive = Trial Longer)",
    y=0.93,
    fontsize=14
)

plt.tight_layout(rect=[0, 0, 0.95, 0.95])
plt.show()

Average Percent Departure from Guidelines (by CHC, OL, and Race)

In [ ]:

df = sent_df[[
    "CHC", "OL", "RACE",
    "GL_MIN", "GL_MAX",
    "MNTHS_VARIANCE/DEPARTURE"
]].dropna().copy()

df = df[(df["GL_MIN"] > 0) & (df["GL_MAX"] > 0)].copy()

# # rmv outliers
# q1 = df["MNTHS_VARIANCE/DEPARTURE"].quantile(0.25)
# q3 = df["MNTHS_VARIANCE/DEPARTURE"].quantile(0.75)
# iqr = q3 - q1

# lower = q1 - 1.5 * iqr
# upper = q3 + 1.5 * iqr

# df = df[
#     (df["MNTHS_VARIANCE/DEPARTURE"] >= lower) &
#     (df["MNTHS_VARIANCE/DEPARTURE"] <= upper)
# ].copy()


df["pct_diff"] = np.nan

# positive departures
mask_pos = df["MNTHS_VARIANCE/DEPARTURE"] > df["GL_MAX"]
df.loc[mask_pos, "pct_diff"] = (
    (df.loc[mask_pos, "MNTHS_VARIANCE/DEPARTURE"] - df.loc[mask_pos, "GL_MAX"]) 
    / df.loc[mask_pos, "GL_MAX"]
)

# negative departures
mask_neg = df["MNTHS_VARIANCE/DEPARTURE"] < df["GL_MIN"]
df.loc[mask_neg, "pct_diff"] = (
    (df.loc[mask_neg, "MNTHS_VARIANCE/DEPARTURE"] - df.loc[mask_neg, "GL_MIN"]) 
    / df.loc[mask_neg, "GL_MIN"]
)

# keep only true departures
df = df[df["pct_diff"].notna()].copy()

# bin ol into groups of 3
df["OL_bin"] = (df["OL"] // 3) * 3


agg = (
    df.groupby(["CHC", "OL_bin", "RACE"])["pct_diff"]
    .mean()
    .reset_index()
)

chc_vals = sorted(agg["CHC"].unique())
races = sorted(agg["RACE"].unique())

fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharey=True)
axes = axes.flatten()

legend_handles = {}

for i, (ax, chc) in enumerate(zip(axes, chc_vals)):
    sub = agg[agg["CHC"] == chc]

    for race in races:
        race_df = sub[sub["RACE"] == race].sort_values("OL_bin")

        if len(race_df) == 0:
            continue

        line, = ax.plot(
            race_df["OL_bin"],
            race_df["pct_diff"],
            linewidth=2
        )

        if race not in legend_handles:
            legend_handles[race] = line

    ax.set_title(f"CHC {int(chc)}")

    if i in [3, 4, 5]:
        ax.set_xlabel("Offense Level (binned by 3)")
    if i in [0, 3]:
        ax.set_ylabel("Avg % Departure from Guidelines")
    ax.grid(False)

axes[0].set_ylabel("Avg % Departure from Guidelines")


fig.legend(
    legend_handles.values(),
    legend_handles.keys(),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.96),
    ncol=len(legend_handles),
    frameon=False
)

plt.suptitle(
    "Average Percent Departure from Guidelines (by CHC, OL, and Race)",
    y=1,
    fontsize=13
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Average % deparute by offense level and different demographic variables.

This graph is similar to the one above except it does not differentiate different CHC values.

In [ ]:
df = sent_df[[
    "OL", "RACE", "SEX", "AGE", "EDUCATION",
    "GL_MIN", "GL_MAX",
    "MNTHS_VARIANCE/DEPARTURE"
]].dropna().copy()

df = df[(df["GL_MIN"] > 0) & (df["GL_MAX"] > 0)].copy()

# # remove outliers
# q1 = df["MNTHS_VARIANCE/DEPARTURE"].quantile(0.25)
# q3 = df["MNTHS_VARIANCE/DEPARTURE"].quantile(0.75)
# iqr = q3 - q1

# lower = q1 - 1.5 * iqr
# upper = q3 + 1.5 * iqr

# df = df[
#     (df["MNTHS_VARIANCE/DEPARTURE"] >= lower) &
#     (df["MNTHS_VARIANCE/DEPARTURE"] <= upper)
# ].copy()

# dept %
df["pct_diff"] = np.nan

# Above-guideline departures: departure amount / GL_MAX
mask_pos = df["MNTHS_VARIANCE/DEPARTURE"] > 0
df.loc[mask_pos, "pct_diff"] = (
    df.loc[mask_pos, "MNTHS_VARIANCE/DEPARTURE"] /
    df.loc[mask_pos, "GL_MAX"]
)

# Below-guideline departures: departure amount / GL_MIN
mask_neg = df["MNTHS_VARIANCE/DEPARTURE"] < 0
df.loc[mask_neg, "pct_diff"] = (
    df.loc[mask_neg, "MNTHS_VARIANCE/DEPARTURE"] /
    df.loc[mask_neg, "GL_MIN"]
)

df = df[df["pct_diff"].notna()].copy()

# bin ol
df["OL_bin"] = ((df["OL"] - 1) // 3) * 3 + 1

age_bins = list(range(16, 96, 15)) 
age_labels = [f"{age_bins[i]}–{age_bins[i+1]-1}" for i in range(len(age_bins)-1)]

df["AGE_BIN"] = pd.cut(
    df["AGE"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True
)

# aggregate function for reuse
def make_agg(data, group_col):
    return (
        data.groupby(["OL_bin", group_col], observed=False)["pct_diff"]
        .mean()
        .reset_index()
    )
agg_race = make_agg(df, "RACE")
agg_sex = make_agg(df, "SEX")
agg_age = make_agg(df, "AGE_BIN")
agg_edu = make_agg(df, "EDUCATION")


def plot_panel(ax, agg, group_col, title, order=None):
    if order is None:
        groups = [g for g in agg[group_col].dropna().unique()]
        try:
            groups = sorted(groups)
        except TypeError:
            groups = list(groups)
    else:
        groups = [g for g in order if g in set(agg[group_col].dropna())]

    for group in groups:
        sub = agg[agg[group_col] == group].sort_values("OL_bin")
        if len(sub) == 0:
            continue

        ax.plot(
            sub["OL_bin"],
            sub["pct_diff"],
            linewidth=2,
            label=str(group)
        )

    ax.axhline(0, linestyle="--", linewidth=1, color="black")
    ax.set_title(title)
    ax.set_xlabel("Offense Level")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize="small")

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9
})

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.flatten()

xticks = sorted(df["OL_bin"].unique())
xtick_labels = [f"{int(x)}–{int(min(x+2, 43))}" for x in xticks]

race_order = ["White", "Black", "Hispanic", "Other"]
sex_order = ["Male", "Female"]
age_order = age_labels

# Education: uses observed sorted values by default
plot_panel(axes[0], agg_race, "RACE", "By Race", order=race_order)
plot_panel(axes[1], agg_sex, "SEX", "By Gender", order=sex_order)
plot_panel(axes[2], agg_age, "AGE_BIN", "By Age Group", order=age_order)
plot_panel(axes[3], agg_edu, "EDUCATION", "By Education")

for ax in axes:
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels, rotation=30)

axes[0].set_ylabel("Avg % Departure from Guidelines")
axes[2].set_ylabel("Avg % Departure from Guidelines")
axes[2].set_xlabel("Offense Level")
axes[3].set_xlabel("Offense Level")

fig.suptitle(
    "Average Percent Departure from Guidelines by Offense Level",
    fontsize=15,
    y=0.98
)

fig.text(
    0.01, -0.02,
    "Note: These graphs do not include immigration cases or offenders sentenced within the guidelines. " + \
    "Percent departure is calculated as the departure amount divided by the relevant guideline bound" + \
    "\n(GL_MIN for below-guideline sentences, GL_MAX for above-guideline sentences). ",
    ha="left",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

Incarceration rate by offense level and different demographic variables.

This graph is the exact same graph as the one above, just with a different sentencing outcome variable.

In [ ]:
df = sent_df[[
    "OL", "RACE", "SEX", "AGE", "EDUCATION",
    "RECIEVED_PRSN_FLAG"
]].dropna().copy()

# bin ol and age
df["OL_bin"] = ((df["OL"] - 1) // 3) * 3 + 1

age_bins = list(range(16, 96, 15)) 
age_labels = [f"{age_bins[i]}–{age_bins[i+1]-1}" for i in range(len(age_bins)-1)]

df["AGE_BIN"] = pd.cut(
    df["AGE"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True
)

# aggregate function for reuse
def make_agg(data, group_col):
    return (
        data.groupby(["OL_bin", group_col], observed=False)["RECIEVED_PRSN_FLAG"]
        .mean()
        .mul(100)
        .reset_index(name="incarceration_rate")
    )

agg_race = make_agg(df, "RACE")
agg_sex = make_agg(df, "SEX")
agg_age = make_agg(df, "AGE_BIN")
agg_edu = make_agg(df, "EDUCATION")

# plotting function for each panel / demographic group
def plot_panel(ax, agg, group_col, title, order=None):
    if order is None:
        groups = [g for g in agg[group_col].dropna().unique()]
        try:
            groups = sorted(groups)
        except:
            groups = list(groups)
    else:
        groups = [g for g in order if g in set(agg[group_col].dropna())]

    for group in groups:
        sub = agg[agg[group_col] == group].sort_values("OL_bin")

        if len(sub) == 0:
            continue

        ax.plot(
            sub["OL_bin"],
            sub["incarceration_rate"],
            linewidth=2,
            label=str(group)
        )

    ax.set_title(title)
    ax.set_xlabel("Offense Level")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize="small")

# plotting
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9
})

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.flatten()

xticks = sorted(df["OL_bin"].unique())
xtick_labels = [f"{int(x)}–{int(min(x+2, 43))}" for x in xticks]

race_order = ["White", "Black", "Hispanic", "Other"]
sex_order = ["Male", "Female"]
age_order = age_labels

plot_panel(axes[0], agg_race, "RACE", "Incarceration Rate by Race", order=race_order)
plot_panel(axes[1], agg_sex, "SEX", "Incarceration Rate by Gender", order=sex_order)
plot_panel(axes[2], agg_age, "AGE_BIN", "Incarceration Rate by Age Group", order=age_order)
plot_panel(axes[3], agg_edu, "EDUCATION", "Incarceration Rate by Education")

for ax in axes:
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels, rotation=30)

axes[0].set_ylabel("Incarceration Rate (%)")
axes[2].set_ylabel("Incarceration Rate (%)")

fig.suptitle(
    "Incarceration Rates by Offense Level Across Demographics",
    fontsize=15,
    y=1.02
)

fig.text(
    0.01, -0.02,
    "Note: In this case, incarceration rate is defined as the percentage of offenders receiving a prison sentence, regardless of probation eligibility. " + \
    "These graphs do not include immigration cases or offenders with missing demographic information.",
    ha="left",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

District-level sentencing outcomes by % of judges in district that are republican-appointed


In [ ]:
df = sent_df[[
    "DIST_CRT",
    "FISCAL_YR",
    "PCT_REPUBLICAN",
    "RCVD_PRIS_SENT_ELGB_PROBAT",
    "MNTHS_PRSN_NO_ALT"
]].copy()

# aggregate to district × fiscal year level
agg = (
    df.groupby(["DIST_CRT", "FISCAL_YR"], observed=False)
    .agg(
        pct_rep=("PCT_REPUBLICAN", "mean"),  # all district-year observations have the same pct_rep value, so mean is fine
        incarceration_rate=("RCVD_PRIS_SENT_ELGB_PROBAT", "mean"),
        avg_sentence=("MNTHS_PRSN_NO_ALT", lambda x: x[x > 0].mean())
    )
    .reset_index()
)

# convert incarceration to percent
agg["incarceration_rate"] *= 100

# drop missing
agg = agg.dropna(subset=["pct_rep", "incarceration_rate", "avg_sentence"])

# Plot
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# ---- Left: Incarceration Rate ----
sns.regplot(
    data=agg,
    x="pct_rep",
    y="incarceration_rate",
    ax=axes[0],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[0].set_title("Incarceration Rate vs. % Republican-Appointed Judges")
axes[0].set_xlabel("% Republican-Appointed Judges")
axes[0].set_ylabel("Incarceration Rate (%)")
axes[0].grid(False)

# ---- Right: Sentence Length ----
sns.regplot(
    data=agg,
    x="pct_rep",
    y="avg_sentence",
    ax=axes[1],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[1].set_title("Avg Sentence Length vs. % Republican-Appointed Judges")
axes[1].set_xlabel("% Republican-Appointed Judges")
axes[1].set_ylabel("Avg Sentence Length (Months)")
axes[1].grid(False)

# remove top and right spines
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "District-Level Sentencing Outcomes vs. Political Composition of Judges",
    fontsize=14,
    y=1.05
)

fig.text(
    0.01, -0.1,
    "Notes: Each point represents a district-year observation. The regression line is fitted using locally weighted least squares (LOWESS) and does not account for any confounders or the fact that the same districts. " \
    "\nappear multiple times. These plots are purely descriptive and should not be interpreted as showing any kind of causal relationship. Additionally, just because a judge was appointed by a Republican president does " \
    "\nnot necessarily mean they are more conservative in their sentencing decisions.",
    ha="left",
    fontsize=9
)

plt.show()

In [ ]:
df = sent_df[[
    "DIST_CRT",
    "FISCAL_YR",
    "PCT_WHITE",
    "RCVD_PRIS_SENT_ELGB_PROBAT",
    "MNTHS_PRSN_NO_ALT"
]].copy()

# aggregate to district × fiscal year level
agg = (
    df.groupby(["DIST_CRT", "FISCAL_YR"], observed=False)
    .agg(
        pct_white=("PCT_WHITE", "mean"),  # all district-year observations have the same pct_white value, so mean is fine
        incarceration_rate=("RCVD_PRIS_SENT_ELGB_PROBAT", "mean"),
        avg_sentence=("MNTHS_PRSN_NO_ALT", lambda x: x[x > 0].mean())
    )
    .reset_index()
)

# convert incarceration to percent
agg["incarceration_rate"] *= 100

# drop missing
agg = agg.dropna(subset=["pct_white", "incarceration_rate", "avg_sentence"])

# Plot
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# ---- Left: Incarceration Rate ----
sns.regplot(
    data=agg,
    x="pct_white",
    y="incarceration_rate",
    ax=axes[0],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[0].set_title("Incarceration Rate vs. % Republican-Appointed Judges")
axes[0].set_xlabel("% Republican-Appointed Judges")
axes[0].set_ylabel("Incarceration Rate (%)")
axes[0].grid(False)

# ---- Right: Sentence Length ----
sns.regplot(
    data=agg,
    x="pct_white",
    y="avg_sentence",
    ax=axes[1],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[1].set_title("Avg Sentence Length vs. % of Judges on Bench that are White")
axes[1].set_xlabel("% White Population")
axes[1].set_ylabel("Avg Sentence Length (Months)")
axes[1].grid(False)

# remove top and right spines
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "District-Level Sentencing Outcomes vs. Racial Composition of Judges",
    fontsize=14,
    y=1.05
)

fig.text(
    0.01, -0.1,
    "Notes: Each point represents a district-year observation. The regression line is fitted using locally weighted least squares (LOWESS) and does not account for any confounders or the fact that the same districts. " \
    "\nappear multiple times. These plots are purely descriptive and should not be interpreted as showing any kind of causal relationship. Additionally, just because a judge was appointed by a Republican president does " \
    "\nnot necessarily mean they are more conservative in their sentencing decisions.",
    ha="left",
    fontsize=9
)

plt.show()

In [ ]:
df = sent_df[[
    "DIST_CRT",
    "FISCAL_YR",
    "PCT_MALE",
    "RCVD_PRIS_SENT_ELGB_PROBAT",
    "MNTHS_PRSN_NO_ALT"
]].copy()

df["PCT_FEMALE"] = 1 - df["PCT_MALE"]

# aggregate to district × fiscal year level
agg = (
    df.groupby(["DIST_CRT", "FISCAL_YR"], observed=False)
    .agg(
        pct_female=("PCT_FEMALE", "mean"),  # all district-year observations have the same pct_female value, so mean is fine
        incarceration_rate=("RCVD_PRIS_SENT_ELGB_PROBAT", "mean"),
        avg_sentence=("MNTHS_PRSN_NO_ALT", lambda x: x[x > 0].mean())
    )
    .reset_index()
)

# convert incarceration to percent
agg["incarceration_rate"] *= 100

# drop missing
agg = agg.dropna(subset=["pct_female", "incarceration_rate", "avg_sentence"])

# Plot
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# ---- Left: Incarceration Rate ----
sns.regplot(
    data=agg,
    x="pct_female",
    y="incarceration_rate",
    ax=axes[0],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[0].set_title("Incarceration Rate vs. % Female Judges")
axes[0].set_xlabel("% Female Judges")
axes[0].set_ylabel("Incarceration Rate (%)")
axes[0].grid(False)

# ---- Right: Sentence Length ----
sns.regplot(
    data=agg,
    x="pct_female",
    y="avg_sentence",
    ax=axes[1],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[1].set_title("Avg Sentence Length vs. % Female Judges")
axes[1].set_xlabel("% Female Judges")
axes[1].set_ylabel("Avg Sentence Length (Months)")
axes[1].grid(False)

# remove top and right spines
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "District-Level Sentencing Outcomes vs. % Female Judges",
    fontsize=14,
    y=1.05
)

fig.text(
    0.01, -0.1,
    "Notes: Each point represents a district-year observation. The regression line is fitted using locally weighted least squares (LOWESS) and does not account for any confounders or the fact that the same districts. " \
    "\nappear multiple times. These plots are purely descriptive and should not be interpreted as showing any kind of causal relationship. Additionally, just because a judge was appointed by a Republican president does " \
    "\nnot necessarily mean they are more conservative in their sentencing decisions.",
    ha="left",
    fontsize=9
)

plt.show()

In [ ]:
df = sent_df[[
    "DIST_CRT",
    "FISCAL_YR",
    "REP_PCT_VOTE",
    "RCVD_PRIS_SENT_ELGB_PROBAT",
    "MNTHS_PRSN_NO_ALT"
]].copy()

# aggregate to district × fiscal year level
agg = (
    df.groupby(["DIST_CRT", "FISCAL_YR"], observed=False)
    .agg(
        rep_pct_vote=("REP_PCT_VOTE", "mean"),  # all district-year observations have the same rep_pct_vote value, so mean is fine
        incarceration_rate=("RCVD_PRIS_SENT_ELGB_PROBAT", "mean"),
        avg_sentence=("MNTHS_PRSN_NO_ALT", lambda x: x[x > 0].mean())
    )
    .reset_index()
)

# convert incarceration to percent
agg["incarceration_rate"] *= 100

# drop missing
agg = agg.dropna(subset=["rep_pct_vote", "incarceration_rate", "avg_sentence"])

# Plot
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# ---- Left: Incarceration Rate ----
sns.regplot(
    data=agg,
    x="rep_pct_vote",
    y="incarceration_rate",
    ax=axes[0],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[0].set_title("Incarceration Rate vs. % Republican Vote")
axes[0].set_xlabel("% Republican Vote")
axes[0].set_ylabel("Incarceration Rate (%)")
axes[0].grid(False)

# ---- Right: Sentence Length ----
sns.regplot(
    data=agg,
    x="rep_pct_vote",
    y="avg_sentence",
    ax=axes[1],
    scatter_kws={"alpha": 0.2, "s": 15},
    line_kws={"color": "black"},
    lowess=True
)

axes[1].set_title("Avg Sentence Length vs. % Republican Vote")
axes[1].set_xlabel("% Republican Vote")
axes[1].set_ylabel("Avg Sentence Length (Months)")
axes[1].grid(False)

# remove top and right spines
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "District-Level Sentencing Outcomes vs. % Republican Vote",
    fontsize=14,
    y=1.05
)

fig.text(
    0.01, -0.1,
    "Notes: Each point represents a district-year observation. The regression line is fitted using locally weighted least squares (LOWESS) and does not account for any confounders or the fact that the same districts. " \
    "\nappear multiple times. The % Republican vote comes from an interpolation of presidential election results (from MIT Election Lab) and may not perfectly capture the political leanings of the district in a given year.",
    ha="left",
    fontsize=9
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# ---------------------------
# 1. Prepare data
# ---------------------------
df = sent_df[["OL", "SEX"]].dropna().copy()
df = df[df["SEX"].isin(["Male", "Female"])]

male_ol = df.loc[df["SEX"] == "Male", "OL"]
female_ol = df.loc[df["SEX"] == "Female", "OL"]

# ---------------------------
# 2. KS test
# ---------------------------
ks_stat, p_value = ks_2samp(male_ol, female_ol)

def sig_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

stars = sig_stars(p_value)

# ---------------------------
# 3. Bin offense levels
# ---------------------------
bins = np.arange(1, 46, 3)  # 1-3, 4-6, ..., 43-45

# normalized weights so each histogram sums to 1
weights_m = np.ones(len(male_ol)) / len(male_ol)
weights_f = np.ones(len(female_ol)) / len(female_ol)

# ---------------------------
# 4. Plot
# ---------------------------
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11
})

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.hist(
    male_ol,
    bins=bins,
    weights=weights_m,
    histtype="step",
    linewidth=2,
    label="Male"
)

ax.hist(
    female_ol,
    bins=bins,
    weights=weights_f,
    histtype="step",
    linewidth=2,
    label="Female"
)

# x tick labels
xticks = bins[:-1]
xtick_labels = [f"{int(x)}–{int(min(x+2, 43))}" for x in xticks]
ax.set_xticks(xticks + 1)
ax.set_xticklabels(xtick_labels, rotation=30)

ax.set_xlabel("Offense Level Bin")
ax.set_ylabel("Proportion")
ax.set_title("Distribution of Offense Severity by Gender")
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)

# KS annotation
annotation_text = f"KS = {ks_stat:.3f}{stars}\np = {p_value:.3g}"
ax.text(
    0.98, 0.98,
    annotation_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8)
)

fig.text(
    0.01, -0.02,
    "Note: Each histogram is normalized to sum to 1 across bins. "
    "Asterisks denote KS-test significance: * p<0.05, ** p<0.01, *** p<0.001.",
    ha="left",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

In [ ]:
df = sent_df[["OL", "RACE"]].dropna().copy()

race_order = ["White", "Black", "Hispanic", "Other"]
df = df[df["RACE"].isin(race_order)]

bins = np.arange(1, 44, 3) 
xticks = bins[:-1]
xtick_labels = [f"{int(x)}–{int(min(x+2, 43))}" for x in xticks]

# consistent colors
colors = {
    "White": "#4C72B0",
    "Black": "#DD8452",
    "Hispanic": "#55A868",
    "Other": "#8172B2"
}

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 10
})

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)
axes = axes.flatten()

for i, race in enumerate(race_order):
    ax = axes[i]
    sub = df.loc[df["RACE"] == race, "OL"]

    if len(sub) == 0:
        continue

    weights = np.ones(len(sub)) / len(sub)
    color = colors[race]

    ax.hist(
        sub,
        bins=bins,
        weights=weights,
        color=color,
        alpha=0.25,
        edgecolor=color,
        linewidth=1.5
    )
    ax.hist(
        sub,
        bins=bins,
        weights=weights,
        histtype="step",
        color=color,
        linewidth=2
    )

    # summary stats
    median_val = np.median(sub)
    q1, q3 = np.percentile(sub, [25, 75])
    iqr_val = q3 - q1
    std_val = np.std(sub, ddof=1)
    kurt_val = kurtosis(sub, fisher=False, bias=False)  # Pearson kurtosis

    # median line
    ax.axvline(
        median_val,
        color=color,
        linestyle="--",
        linewidth=2
    )

    # IQR shaded span
    ax.axvspan(
        q1, q3,
        color=color,
        alpha=0.10
    )

    # stats box
    stats_text = (
        f"Median: {median_val:.1f}\n"
        f"IQR: {iqr_val:.1f}\n"
        f"SD: {std_val:.1f}\n"
        f"Kurtosis: {kurt_val:.2f}"
    )

    ax.text(
        0.97, 0.95,
        stats_text,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85)
    )

    ax.set_title(race)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

# shared ticks
for ax in axes:
    ax.set_xticks(xticks + 1)
    ax.set_xticklabels(xtick_labels, rotation=30)

# axis labels
axes[2].set_xlabel("Offense Level Bin")
axes[3].set_xlabel("Offense Level Bin")
axes[0].set_ylabel("Proportion")
axes[2].set_ylabel("Proportion")

fig.suptitle(
    "Distribution of Offense Severity by Race",
    fontsize=16,
    y=0.99
)

fig.text(
    0.01, 0.01,
    "Note: Histograms are normalized within race so each panel sums to 1. "
    "Dashed vertical lines indicate medians; shaded bands indicate the interquartile range.",
    ha="left",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.show()

In [ ]:
# box plot of MNTHS_PRSN_NO_ALT by OL, with hue that denotes how many observations are in each OL

df = sent_df[sent_df["MNTHS_PRSN_NO_ALT"] > 0.03].copy()  # filter out zero-month sentences for better log scale visualization

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,
    x="OL",
    y="MNTHS_PRSN_NO_ALT",
    hue=df["OL"].map(df["OL"].value_counts()),
    palette="Blues",
    showfliers=False,
    log_scale=True
)

# make xticks integers
plt.xticks(ticks=range(0, 43), labels=range(1, 44))
plt.title("Distribution of Log Sentence Length by Offense Level")
plt.xlabel("Offense Level")
plt.ylabel("Log Sentence Length (Months)")
plt.legend(title="Count in OL", bbox_to_anchor=(0.01, 1), loc="upper left")
plt.tight_layout()

# add notes about outliers and counts to plot
plt.figtext(
    0.06, -0.07,
    "Note: The color intensity of each box corresponds to the number of observations in that offense level (darker = more observations). Outliers are not shown in this plot. " \
    "\nAdditionally, immigration cases are not included in this graph.",
    ha="left",
    fontsize=9
)

plt.show()

In [ ]:
df = (
    sent_df[sent_df["MNTHS_PRSN_NO_ALT"]>0.03]
    .sort_values(by="OL") # sort by OL so that animation frames are in order
).copy()

df["MNTHS_PRSN_NO_ALT"] = np.log(df["MNTHS_PRSN_NO_ALT"].clip(upper=470))

fig = px.histogram(
    df, 
    x="MNTHS_PRSN_NO_ALT", 
    animation_frame="OL", 
    nbins=40, 
    title="Distribution of Prison Sentence (Months) by Offense Level",
    labels={"MNTHS_PRSN_NO_ALT": "Prison Sentence (Months)", "count": "Percent"},
    histnorm="percent",
    range_x=(0, 6.2),
    range_y=(0, 100),
    marginal='rug',
    height=600,
    width=800
)

fig.update_layout(xaxis_title="Prison Sentence (Months)", yaxis_title="Percent of Offenders")
fig.update_layout(margin={"r": 20, "t": 50, "l": 60, "b": 0})
fig.show()

Heatmap showing different summary statistics for each cell of the sentencing table

In [ ]:
def plot_ol_chc_heatmap(
    df: pd.DataFrame,
    value_col: str,
    ol_col: str = "OL",
    chc_col: str = "CHC",
    stat: str | float = "count",
    figsize: tuple = (12, 8),
    cmap: str = "YlGnBu",
    annot: bool = True,
    fmt: str | None = None,
    mask_na: bool = False,
    sort_numeric: bool = True,
    title: str | None = None,
    cbar_kws: dict | None = None,
) -> pd.DataFrame:
    """
    Plot an OL x CHC heatmap for a chosen summary statistic.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    value_col : str
        Column to summarize within each OL x CHC cell.
    ol_col : str, default "OL"
        Offense level column.
    chc_col : str, default "CHC"
        Criminal history category column.
    stat : {"count", "mean", "std", "median"} or float
        Summary statistic to compute.
        If float in [0, 1], computes that quantile.
    figsize : tuple, default (12, 8)
        Figure size.
    cmap : str, default "YlGnBu"
        Matplotlib/seaborn colormap.
    annot : bool, default True
        Whether to annotate cells.
    fmt : str or None, default None
        Annotation format. If None, chosen automatically.
    mask_na : bool, default False
        If True, mask NA cells in the heatmap.
    sort_numeric : bool, default True
        If True, sort OL/CHC numerically when possible.
    title : str or None, default None
        Plot title. If None, generated automatically.
    cbar_kws : dict or None
        Passed to seaborn.heatmap() colorbar options.

    Returns
    -------
    pd.DataFrame
        The pivot table used for plotting.
    """

    if value_col not in df.columns:
        raise ValueError(f"{value_col!r} not found in dataframe.")
    if ol_col not in df.columns:
        raise ValueError(f"{ol_col!r} not found in dataframe.")
    if chc_col not in df.columns:
        raise ValueError(f"{chc_col!r} not found in dataframe.")

    work = df[[ol_col, chc_col, value_col]].copy()

    # Drop rows missing OL/CHC. For count, value_col can be missing and still counted if desired,
    # but here we count non-missing value_col entries for consistency.
    work = work.dropna(subset=[ol_col, chc_col])

    # Try to sort OL/CHC numerically for nicer display
    def maybe_numeric_sort(vals):
        vals = pd.Index(vals)
        if not sort_numeric:
            return vals
        try:
            numeric_vals = pd.to_numeric(vals)
            return vals[np.argsort(numeric_vals)]
        except Exception:
            return vals.sort_values()

    # Choose aggregator
    if stat == "count":
        grouped = (
            work.groupby([ol_col, chc_col], observed=False)[value_col]
            .count()
            .reset_index(name="stat_value")
        )
        stat_label = "Count"

    elif stat == "mean":
        grouped = (
            work.groupby([ol_col, chc_col], observed=False)[value_col]
            .mean()
            .reset_index(name="stat_value")
        )
        stat_label = f"Mean {value_col}"

    elif stat == "std":
        grouped = (
            work.groupby([ol_col, chc_col], observed=False)[value_col]
            .std()
            .reset_index(name="stat_value")
        )
        stat_label = f"Std. Dev. of {value_col}"

    elif isinstance(stat, (float, int)) and 0 <= float(stat) <= 1:
        q = float(stat)
        grouped = (
            work.groupby([ol_col, chc_col], observed=False)[value_col]
            .quantile(q)
            .reset_index(name="stat_value")
        )
        stat_label = f"{q:.0%} Quantile of {value_col}"

    else:
        raise ValueError(
            "stat must be one of {'count', 'mean', 'std'} "
            "or a float between 0 and 1 for quantiles."
        )

    pivot = grouped.pivot(index=ol_col, columns=chc_col, values="stat_value")

    pivot = pivot.reindex(index=maybe_numeric_sort(pivot.index))
    pivot = pivot.reindex(columns=maybe_numeric_sort(pivot.columns))

    if fmt is None:
        if stat == "count":
            fmt = ".0f"
        elif stat == "std":
            fmt = ".2f"
        else:
            fmt = ".2f"

    if title is None:
        title = f"{stat_label} by {ol_col} × {chc_col}"

    plt.figure(figsize=figsize)

    sns.heatmap(
        pivot,
        cmap=cmap,
        annot=annot,
        fmt=fmt,
        linewidths=0.5,
        linecolor="white",
        mask=pivot.isna() if mask_na else None,
        cbar=False,
        cbar_kws=cbar_kws if cbar_kws is not None else {"label": stat_label},
    )

    plt.yticks(ticks=np.arange(len(pivot.index)) + 0.5, labels=pivot.index.astype(int), ha="right")
    plt.xticks(ticks=np.arange(len(pivot.columns)) + 0.5, labels=pivot.columns.astype(int), ha="right")
    plt.title(title)
    plt.xlabel(chc_col)
    plt.ylabel(ol_col)
    plt.tight_layout()
    plt.show()

plot_ol_chc_heatmap(sent_df, value_col="MNTHS_PRSN_NO_ALT", stat="mean", figsize=(7, 10), cmap="Purples")

In [ ]:
# histogram of Z

df = sent_df[(sent_df["STAT_MIN"] < sent_df["GL_MAX"])].copy()  # filter out zero-month sentences and extreme outliers for better visualization

cell_stats = df.groupby(["OL", "CHC"])["MNTHS_PRSN_NO_ALT"].agg(["mean", "std"])
df = df.merge(cell_stats, on=["OL", "CHC"], how="left")
df["Z"] = (
    df["MNTHS_PRSN_NO_ALT"] - df["mean"]    
) / df["std"]

plt.figure(figsize=(10, 6))
sns.histplot(df["Z"], bins=100, kde=True, color="#4C72B0")
plt.title("Histogram of Standardized Sentence Length (Z)")  
plt.xlabel("Z-score of Sentence Length")
plt.ylabel("Count")
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
df = sent_df[[
    "OL", "CHC",
    "MNTHS_PRSN_NO_ALT",
    "RECIEVED_PRSN_FLAG"
]].dropna().copy()

df = df[df["RECIEVED_PRSN_FLAG"].isin([0, 1])].copy()

# calculate aggregate statisitcs
agg = (
    df.groupby(["OL", "CHC"])
    .agg(
        avg_sentence=("MNTHS_PRSN_NO_ALT", "mean"),
        incarceration_rate=("RECIEVED_PRSN_FLAG", "mean")
    )
    .reset_index()
)

# custom colormap
cmap = LinearSegmentedColormap.from_list(
    "custom_blues",
    ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]  # light → dark blue
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# ----- Left: Avg sentence -----
sc1 = axes[0].scatter(
    agg["OL"],
    agg["CHC"],
    c=agg["avg_sentence"],
    cmap=cmap,
    s=50
)

axes[0].set_title("Average Sentence Length (Months)")
axes[0].set_xlabel("Offense Level (OL)")
axes[0].set_ylabel("CHC")

cbar1 = plt.colorbar(sc1, ax=axes[0])
cbar1.set_label("Avg Months")

# ----- Right: Incarceration rate -----
sc2 = axes[1].scatter(
    agg["OL"],
    agg["CHC"],
    c=agg["incarceration_rate"],
    cmap=cmap,
    s=50
)

axes[1].set_title("Incarceration Rate")
axes[1].set_xlabel("Offense Level (OL)")

cbar2 = plt.colorbar(sc2, ax=axes[1])
cbar2.set_label("Rate")

for ax in axes:
    ax.grid(False)
    ax.set_xlim(0.5, 43.5)   
    ax.set_ylim(0.5, 6.5)   

plt.suptitle("Sentencing Outcomes by Offense Level and CHC", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
sent_df["STAT_MIN"] = pd.to_numeric(sent_df["STAT_MIN"].replace({"Life": 470}))
sent_df["STAT_MAX"] = pd.to_numeric(sent_df["STAT_MAX"].replace({"Life": 470}))
sent_df["GL_MIN_UNTRUMPED"] = pd.to_numeric(sent_df["GL_MIN_UNTRUMPED"].replace({"Life": 470}))
sent_df["GL_MAX_UNTRUMPED"] = pd.to_numeric(sent_df["GL_MAX_UNTRUMPED"].replace({"Life": 470}))

In [ ]:
# histogram of Z

df = sent_df[
    (sent_df["STAT_MIN"] <= sent_df["GL_MIN_UNTRUMPED"]) &
    (sent_df["STAT_MAX"] >= sent_df["GL_MAX_UNTRUMPED"]) & 
    (sent_df["MNTHS_PRSN_NO_ALT"] > 0.03) 
].copy()

df["MNTHS_PRSN_NO_ALT"] = df["MNTHS_PRSN_NO_ALT"].clip(upper=470)  

cell_stats = df.groupby(["OL", "CHC"])["MNTHS_PRSN_NO_ALT"].agg(["mean", "std"])
df = df.merge(cell_stats, on=["OL", "CHC"], how="left")
df["Z"] = (
    df["MNTHS_PRSN_NO_ALT"] - df["mean"]    
) / df["std"]

plt.figure(figsize=(10, 6))
sns.histplot(df["Z"], bins=100, kde=True, color="#4C72B0")
plt.title("Histogram of Standardized Sentence Length (Z)")  
plt.xlabel("Z-score of Sentence Length")
plt.ylabel("Count")
plt.grid(False)
plt.tight_layout()
plt.show()